# LangChain 

## What is LangChain?

LangChain is a **framework** that makes it easy to build applications powered by Large Language Models (LLMs).

Instead of writing raw API calls, LangChain gives you clean, reusable building blocks:

| Building Block | What it does |
|---|---|
| **Model** | Connects to the AI (e.g. Groq, OpenAI, Anthropic) |
| **Prompt Template** | Structures your message before sending to the AI |
| **Output Parser** | Converts the AI response into a plain Python string |
| **Chain** | Links everything together with the `|` pipe operator |

## What you will learn

1. Set up the environment
2. Connect to an AI model
3. Send your first message
4. Use Prompt Templates
5. Parse the output cleanly
6. Build a Chain with `|`
7. Stream tokens live
8. Build a multi-turn conversation

---
> **Run every cell from top to bottom.** Each step builds on the previous one.

## Step 1 — Check the Kernel

Before anything, let's confirm the Jupyter kernel is running correctly.

Running `1 + 1` should output `2`.

> This is just a sanity check — nothing LangChain-specific yet.

In [20]:
1 + 1

2

## Step 2 — Import LangChain and Check the Version

`langchain` is the main library. Let's verify it is installed and see the version number.

> If this fails with `ModuleNotFoundError`, run `pip install langchain` in your terminal.

In [21]:
import langchain

print("LangChain version:", langchain.__version__)

LangChain version: 1.3.11


## Step 3 — Load the API Key from `.env`

We need a **GROQ_API_KEY** to talk to the Groq AI service.

**Why a `.env` file instead of pasting the key in your code?**
- Keeps secrets out of your source files
- If you share the notebook, nobody sees your key
- Industry standard — **never hardcode API keys!**

The `.env` file in this folder looks like:
```
GROQ_API_KEY=gsk_your_key_here
```

`python-dotenv` reads that file and loads the key into the environment automatically.

In [22]:
from dotenv import load_dotenv

load_dotenv()   # reads .env and loads GROQ_API_KEY into the environment

print("API key loaded from .env")

API key loaded from .env


## Step 4 — Connect to the AI Model

An **LLM (Large Language Model)** is the AI brain we will talk to.

- **Groq** is the cloud provider — they run the model and give us a fast API
- **qwen/qwen3-32b** is the specific model (32 billion parameters, open-source)

`init_chat_model("groq:model-name")` is LangChain's universal loader.
You can swap `groq` for `openai`, `anthropic`, `ollama`, etc. without changing anything else.

In [23]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")

print("Model loaded:", type(model).__name__)

Model loaded: ChatGroq


## Step 5 — Your First Message to the AI

**`.invoke()`** sends a message to the model and waits for the **complete response**.

- It takes a string (or a list of messages)
- It returns an `AIMessage` object
- We use `.content` to extract the plain text

Think of `.invoke()` as a simple function call: give it a question, get an answer.

In [24]:
response = model.invoke("Hello! What is LangChain in one sentence?")

print(response.content)

<think>
Okay, the user asked for a one-sentence explanation of LangChain. Let me start by recalling what LangChain is. It's a framework, right? Designed for working with language models.

Hmm, what's its main purpose? I think it's about connecting different components, like models, data sources, and user interfaces. Oh, right, it's for building applications with LLMs by chaining together various tools and systems.

Wait, should I mention specific features like agents or memory? Maybe keep it general for a one-sentence answer. The key points are integrating models with data, tools, and applications, and enabling complex workflows.

Let me check if there's an official definition. The LangChain website says it's a framework for developing applications with LLMs by combining models, data, and tools. That's concise. I can rephrase that to make sure it's in one sentence.

So, putting it all together: LangChain is a framework that enables developers to build applications powered by language m

## Step 6 — Look Inside the Response Object

The model does not just return text — it returns an **`AIMessage` object** with extra metadata.

Let's inspect it to understand what is inside.
This matters because later tools (parsers, memory) work with the full object, not just `.content`.

In [25]:
print("Type:", type(response))
print()
print("Full AIMessage object:")
print(response)

Type: <class 'langchain_core.messages.ai.AIMessage'>

Full AIMessage object:
content='<think>\nOkay, the user asked for a one-sentence explanation of LangChain. Let me start by recalling what LangChain is. It\'s a framework, right? Designed for working with language models.\n\nHmm, what\'s its main purpose? I think it\'s about connecting different components, like models, data sources, and user interfaces. Oh, right, it\'s for building applications with LLMs by chaining together various tools and systems.\n\nWait, should I mention specific features like agents or memory? Maybe keep it general for a one-sentence answer. The key points are integrating models with data, tools, and applications, and enabling complex workflows.\n\nLet me check if there\'s an official definition. The LangChain website says it\'s a framework for developing applications with LLMs by combining models, data, and tools. That\'s concise. I can rephrase that to make sure it\'s in one sentence.\n\nSo, putting it all

## Step 7 — Prompt Templates: Give the AI a Role

So far we sent a plain string. In real apps you want to:
1. Set a **system role** — tell the AI how to behave (e.g. "you are a Python tutor")
2. Insert the **user's question** dynamically into a pre-written template

`ChatPromptTemplate` handles both:
- `"system"` — instructions the AI follows (invisible to the end user)
- `"human"` — the actual question, with `{placeholders}` for variables

> **Key idea:** First we *print the formatted prompt* to see exactly what the AI receives, before sending it.

In [26]:
from langchain_core.prompts import ChatPromptTemplate

# Define the template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Python tutor. Explain things simply."),
    ("human", "{question}"),   # {question} is a placeholder filled in at runtime
])

# Fill in the placeholder with a real question and inspect the result
formatted = prompt.invoke({"question": "What is a list?"})

print("Formatted prompt (what the AI receives):")
print(formatted)

Formatted prompt (what the AI receives):
messages=[SystemMessage(content='You are a helpful Python tutor. Explain things simply.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is a list?', additional_kwargs={}, response_metadata={})]


In [27]:
# Now send the formatted prompt to the model
response2 = model.invoke(formatted)

print(response2.content)

<think>
Okay, the user is asking, "What is a list?" Let me think about how to explain this in Python. First, I need to remember that they might be a beginner, so I should keep it simple.

A list in Python is a collection of items. I should mention that it's ordered and changeable. Maybe give an example of how to create a list, like using square brackets. Also, lists can have different data types, so I should highlight that.

Wait, maybe start with the definition: a list is used to store multiple items in a single variable. Then explain the key features—ordered, mutable, allows duplicate elements. Examples would help. Like, creating a list of fruits or numbers.

I should also mention common operations, like accessing elements with indexes, changing items, adding to the list, etc. But not too detailed, just enough to give a basic understanding.

Oh, and compare it to other data structures briefly. For example, lists are different from tuples because they are mutable. But maybe that's mor

## Step 8 — Output Parser: Get Clean Text Back

The model returns an `AIMessage` **object**. In most apps, you just want a plain **string**.

`StrOutputParser` converts the object into a plain Python string.

### Common Confusion — Why does the text look the same?

**The TEXT is the same. The PYTHON TYPE is different.**

| | Without parser | With parser |
|---|---|---|
| **Type** | `AIMessage` (an object) | `str` (a plain string) |
| **To get the text** | `raw.content` (must call `.content`) | `clean` (use directly) |
| **Can concatenate?** |  `raw + " extra"` → **CRASH** | `clean + " extra"` → works |
| **Can pass to other functions?** |  Many functions expect `str` |  Yes, it IS a string |

The parser does **not** change what the AI said — it changes **how Python stores it**.  
This matters the moment you try to use the text anywhere else in your code.

In [34]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()


raw = model.invoke(formatted)


print("  WITHOUT PARSER — what the model actually returns")

print(raw)          # prints the FULL AIMessage object — content + metadata + token counts

print()


clean = parser.invoke(raw)


print("  WITH PARSER — plain text only")

print(clean)        # prints ONLY the text — nothing else

print()



  WITHOUT PARSER — what the model actually returns
content='<think>\nOkay, the user is asking, "What is a list?" Let me start by recalling what I know about Python lists. Lists are one of the most basic and commonly used data structures in Python. They are used to store a collection of items in a single variable. But how do I explain this simply?\n\nFirst, I should define a list in simple terms. Maybe compare it to something familiar, like a shopping list. Then mention that in Python, lists can hold different data types, not just strings. Wait, but in Python, all elements in a list don\'t have to be the same type. That\'s a key point.\n\nI need to mention that lists are ordered, meaning the items have a specific sequence. Also, they are mutable, which means you can change them after creation. Oh, right, unlike strings which are immutable. So mutability is important here.\n\nExamples would help. Maybe show a simple list like [1, 2, 3] and a mixed list like [\'apple\', 42, True]. Then ex

## Step 9 — Chains: The `|` Pipe Operator

A **Chain** links your prompt, model, and parser into one reusable pipeline.

LangChain uses the `|` (pipe) operator — just like Unix shell pipes:

```
question  ->  prompt  ->  model  ->  parser  ->  plain text
```

The `|` means: *take the output of the left side and feed it into the right side.*

This is called **LCEL — LangChain Expression Language**. It is the modern, recommended way to build LangChain apps.

> After building the chain, you call `.invoke()` **once** on the whole chain — not on each piece separately.

In [30]:
# Build the chain: prompt -> model -> parser  (all linked with the | operator)
chain = prompt | model | StrOutputParser()

# One call invokes the entire pipeline
answer = chain.invoke({"question": "What is a Python dictionary?"})

print(answer)

<think>
Okay, the user is asking about Python dictionaries. Let me start by recalling what a dictionary is. A dictionary in Python is a collection of key-value pairs. It's unordered, mutable, and each key must be unique. I should explain that it's used to store data in a way that makes it easy to access by keys rather than indexes like in lists.

Hmm, how to explain this simply? Maybe start by comparing it to a real-world dictionary where you look up a word (key) and find its definition (value). That analogy is common and helpful. Then mention that keys can be any immutable type, like strings or numbers, and values can be any data type.

I should also mention some basic operations, like creating a dictionary using curly braces or the dict() function. Show an example, like {'name': 'Alice', 'age': 25}. Then talk about accessing values with keys, adding or modifying entries, and maybe iterating through a dictionary.

Wait, the user might not know about lists yet. But since they're asking

## Step 10 — Streaming: See Tokens Arrive Live

`.invoke()` waits for the **full response** before showing anything.
`.stream()` sends back **one small chunk (token) at a time** as the AI generates it.

**Why use streaming?**
- Your app feels **faster** — users see text immediately instead of waiting
- Better user experience for long answers
- Required for chatbot-style interfaces (like ChatGPT)

`.stream()` returns a **generator** — we loop over it and print each chunk as it arrives.

In [31]:
print("Streaming response (tokens arrive one by one):")
print()

# .stream() yields chunks as the AI generates them — no waiting for the full response
for chunk in chain.stream({"question": "Explain Python lists in 3 sentences."}):
    print(chunk, end="", flush=True)   # end="" stays on same line; flush=True shows immediately

print()
print("Stream complete.")

Streaming response (tokens arrive one by one):

<think>
Okay, I need to explain Python lists in three sentences. Let me start by recalling what a list is. A list is an ordered collection of items that can be of different types. They are mutable, which means you can change them after creation. How do you create a list? Using square brackets, like [1, 2, 3]. 

Next, I should mention common operations. Like accessing elements with indexes, starting from 0. Also, methods such as append or remove. Maybe mention slicing? But keep it simple.

Third sentence should cover key features. Maybe talk about being heterogeneous, allowing mixed data types, and dynamic size. Also, list comprehensions as a concise way to create lists. Let me check if that's accurate.

Wait, three sentences. First: definition and creation. Second: operations and methods. Third: features like mutability, heterogeneous, dynamic, and comprehensions. That should cover the essentials without being too technical. Let me put it

## Step 11 — Conversation Memory: Multi-Turn Chat

Every `.invoke()` call so far is **stateless** — the AI has no memory of previous messages.

To build a real chatbot, you need to **pass the full conversation history** each time.

LangChain uses a simple list of message objects:
- `SystemMessage` — the system instruction (set once at the start)
- `HumanMessage` — what the user said
- `AIMessage` — what the AI replied

You append each turn to the list and send the whole list every time.
This is exactly what ChatGPT does internally.

In [32]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# The conversation history is just a Python list
history = [
    SystemMessage(content="You are a friendly Python tutor. Keep answers to 2 sentences.")
]

def chat(user_input):
    """Send a message and remember the full conversation."""
    history.append(HumanMessage(content=user_input))
    reply = model.invoke(history)       # pass the FULL history every time
    history.append(AIMessage(content=reply.content))
    return reply.content

# Turn 1
print("User: What is a variable?")
print("AI:", chat("What is a variable?"))
print()

# Turn 2 — the AI remembers the previous question
print("User: Give me an example.")
print("AI:", chat("Give me an example."))

User: What is a variable?
AI: <think>
Okay, the user asked, "What is a variable?" I need to explain this in a simple way. Let's start with the basics. A variable is like a container for data. Maybe use an example they can relate to, like a box holding something. Also, mention that variables have names and values. Should I include how you assign values in Python? Probably, since it's a Python question. So, maybe: "A variable is a named storage location for data. In Python, you assign a value to a variable using the equals sign, like x = 5." That's two sentences and covers the essentials. Check if there's anything else they might need. No, keep it concise. Make sure it's clear and easy to understand.
</think>

A variable is a named storage location for data in a program. In Python, you assign a value to a variable using the equals sign, like `x = 5`.

User: Give me an example.
AI: <think>
Okay, the user asked for an example of a variable in Python. Let me start by recalling that variable

## What's Next?

You have learned the core building blocks of LangChain:

| Concept | What you learned |
|---|---|
| `init_chat_model()` | Connect to any LLM provider |
| `.invoke()` | Get a full response |
| `ChatPromptTemplate` | Give the AI a role and structure your input |
| `StrOutputParser` | Get clean plain text out |
| `chain = prompt | model | parser` | Link steps with `|` into a pipeline |
| `.stream()` | Get tokens as they arrive |
| Message history | Build multi-turn conversations |

---

## Run the Real Apps in This Folder

| File | What it does | How to run |
|---|---|---|
| `flask_app.py` | Simple Q&A web app | `python flask_app.py` then open http://localhost:5000 |
| `flask_stream.py` | Streaming web app | `python flask_stream.py` then open http://localhost:5001 |
| `streamlit_app.py` | Streamlit UI | `streamlit run streamlit_app.py` |

---

## What to Explore Next
- **RAG (Retrieval-Augmented Generation)** — let the AI answer from your own documents
- **Agents and Tools** — let the AI decide which actions to take
- **LangSmith** — trace and debug your chains visually